# 00 - Auditoria e configuracao da atualizacao

Monta o Drive antes do repositorio, inventaria o estado real e prepara o ciclo `20260713`. A gravacao fica bloqueada ate confirmacao explicita.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
ACTIVE_CONFIG_PATH = DATA_ROOT / "operations" / "atualizacao" / "active.json"
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # Opcional: branch, tag ou commit. Vazio usa o default remoto.

os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
for name in ["raw", "checkpoints", "logs", "manifests", "processed", "operations/atualizacao"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("DATA_ROOT:", DATA_ROOT)
print("Repositorio:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

In [ ]:
EXPECTED_CYCLE_ID = "20260713"
CYCLE_DIR = DATA_ROOT / "operations" / "atualizacao" / "ciclos" / EXPECTED_CYCLE_ID
WINDOW = {"data_inicio": "2026-05-01", "data_fim": "2026-07-13"}

COLLECTION_RUNS = [
    {"key": "parlamentares", "lane": "prerequisite", "module": "coleta.parlamentares.collect", "source": "all", "dataset": "parlamentares", "run_id": "prod-atualizacao-20260713-parlamentares", "data_inicio": "2026-05-01", "data_fim": "2026-07-13", "checkpoint_sources": ["camara", "senado"]},
    {"key": "senado_ccj_historico", "lane": "senado", "module": "coleta.senado.ccj_notas.collect", "source": "senado", "dataset": "ccj_notas", "run_id": "prod-historico-senado-ccj", "data_inicio": "1900-01-01", "data_fim": "2026-05-28", "recovery": True},
    {"key": "senado_plenario", "lane": "senado", "module": "coleta.senado.plenario_discursos.collect", "source": "senado", "dataset": "plenario_discursos", "run_id": "prod-atualizacao-20260713-senado-plenario", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
    {"key": "senado_ccj", "lane": "senado", "module": "coleta.senado.ccj_notas.collect", "source": "senado", "dataset": "ccj_notas", "run_id": "prod-atualizacao-20260713-senado-ccj", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
    {"key": "senado_pareceres_pec", "lane": "senado", "module": "coleta.senado.pareceres_pec.collect", "source": "senado", "dataset": "pareceres_pec", "run_id": "prod-atualizacao-20260713-senado-pareceres-pec", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
    {"key": "senado_apartes", "lane": "senado", "module": "coleta.senado.plenario_apartes.collect", "source": "senado", "dataset": "plenario_apartes", "run_id": "prod-atualizacao-20260713-senado-plenario-apartes", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
    {"key": "congresso_textos", "lane": "congresso", "module": "coleta.senado.congresso_discursos.collect", "source": "senado", "dataset": "congresso_discursos", "run_id": "prod-historico-senado-congresso-textos-v1", "data_inicio": "1996-05-01", "data_fim": "2026-07-13"},
    {"key": "camara_ccjc_historico", "lane": "camara_demais", "module": "coleta.camara.ccjc_eventos.collect", "source": "camara", "dataset": "ccjc_eventos", "run_id": "prod-historico-camara-ccjc", "data_inicio": "1900-01-01", "data_fim": "2026-05-28", "recovery": True},
    {"key": "camara_ccjc", "lane": "camara_demais", "module": "coleta.camara.ccjc_eventos.collect", "source": "camara", "dataset": "ccjc_eventos", "run_id": "prod-atualizacao-20260713-camara-ccjc", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
    {"key": "camara_pareceres_pec", "lane": "camara_demais", "module": "coleta.camara.pareceres_pec.collect", "source": "camara", "dataset": "pareceres_pec", "run_id": "prod-atualizacao-20260713-camara-pareceres-pec", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
    {"key": "camara_apartes", "lane": "camara_demais", "module": "coleta.camara.plenario_apartes.collect", "source": "camara", "dataset": "plenario_apartes", "run_id": "prod-atualizacao-20260713-camara-plenario-apartes", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
    {"key": "camara_plenario_historico", "lane": "camara_plenario", "module": "coleta.camara.plenario_discursos.collect", "source": "camara", "dataset": "plenario_discursos", "run_id": "prod-historico-camara-plenario", "data_inicio": "1946-01-01", "data_fim": "2026-05-28", "recovery": True},
    {"key": "camara_plenario", "lane": "camara_plenario", "module": "coleta.camara.plenario_discursos.collect", "source": "camara", "dataset": "plenario_discursos", "run_id": "prod-atualizacao-20260713-camara-plenario", "data_inicio": "2026-05-01", "data_fim": "2026-07-13"},
]

from datetime import datetime, timezone

PROCESSING_RUN_IDS = {
    "parlamentares": "processed-parlamentares-v1-current",
    "textos": "processed-textos-v1-current",
    "parquet": "parquet-textos-v1-current",
    "apartes": "processed-apartes-parlamentares-v1-current",
    "join_audit": "parlamentares-join-20260713",
    "samples": "samples-textos-v1-20260713",
}
EXPECTED_TEXT_PARQUETS = [
    "senado__plenario_discursos.parquet",
    "senado__congresso_discursos.parquet",
    "senado__ccj_notas.parquet",
    "senado__pareceres_pec.parquet",
    "camara__plenario_discursos.parquet",
    "camara__ccjc_eventos.parquet",
    "camara__pareceres_pec.parquet",
]
EXPECTED_TEXT_DATASETS = [name.removesuffix(".parquet").replace("__", "/") for name in EXPECTED_TEXT_PARQUETS]

CONFIG_CANDIDATE = {
    "schema_version": 1,
    "cycle_id": EXPECTED_CYCLE_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "data_root": str(DATA_ROOT),
    "data_inicio": WINDOW["data_inicio"],
    "data_fim": WINDOW["data_fim"],
    "window": WINDOW,
    "historical_floor": "1900-01-01",
    "raw_policy": "immutable_cumulative",
    "processed_policy": "canonical_current",
    "collection_runs": COLLECTION_RUNS,
    "historical_recoveries": [item for item in COLLECTION_RUNS if item.get("recovery")],
    "run_ids": {
        "collection": {item["key"]: item["run_id"] for item in COLLECTION_RUNS},
        "processing": PROCESSING_RUN_IDS,
    },
    "processing_run_ids": PROCESSING_RUN_IDS,
    "expected_text_datasets": EXPECTED_TEXT_DATASETS,
    "expected_text_parquets": EXPECTED_TEXT_PARQUETS,
    "expected_apartes_sources": ["senado", "camara"],
    "expected_processed_bases": ["textos_parlamentares/v1", "parlamentares/v1", "apartes_parlamentares/v1"],
    "expected_processed_outputs": [
        "processed/textos_parlamentares/v1",
        "processed/parlamentares/v1",
        "processed/apartes_parlamentares/v1",
    ],
}
print(json.dumps(CONFIG_CANDIDATE, ensure_ascii=False, indent=2))

## Inventario read-only

Confere pastas, manifests, autosaves, checkpoints, Parquets e locks antes de gravar o ciclo.

In [ ]:
def compact_manifest(path):
    item = read_json(path) or {}
    return {
        "arquivo": path.name,
        "status": item.get("status"),
        "source": item.get("source"),
        "dataset": item.get("dataset"),
        "data_inicio": item.get("data_inicio"),
        "data_fim": item.get("data_fim"),
        "errors": item.get("errors"),
    }

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

import pyarrow.parquet as pq

print("Pasta ativa existe:", DATA_ROOT.exists(), DATA_ROOT)
manifest_paths = [path for path in (DATA_ROOT / "manifests").glob("*.json") if not path.name.endswith(".autosave.json")]
manifests = [read_json(path) for path in manifest_paths]
dataset_cutoffs = {}
for item in manifests:
    if not item or not item.get("dataset") or not item.get("data_fim"):
        continue
    dataset_cutoffs[item["dataset"]] = max(item["data_fim"], dataset_cutoffs.get(item["dataset"], ""))
print("Manifestos finais:", len(manifest_paths))
print("Autosaves:", len(list((DATA_ROOT / "manifests").glob("*.autosave.json"))))
print("Checkpoints:", len(list((DATA_ROOT / "checkpoints").rglob("*.json"))))
parquet_root = DATA_ROOT / "processed" / "textos_parlamentares" / "v1" / "parquet"
parquet_paths = sorted(parquet_root.glob("*.parquet"))
parquet_rows = {path.name: pq.ParquetFile(path).metadata.num_rows for path in parquet_paths}
print("Parquets textuais:", parquet_rows, "total=", sum(parquet_rows.values()))
print("Cortes por dataset:", dataset_cutoffs)
print("Locks ativos:", sorted(path.name for path in (DATA_ROOT / "operations" / "atualizacao" / "locks").glob("*.json")))

known_state = {
    "prod-historico-camara-plenario": read_json(DATA_ROOT / "manifests" / "prod-historico-camara-plenario.autosave.json"),
    "prod-historico-senado-ccj": read_json(DATA_ROOT / "manifests" / "prod-historico-senado-ccj.json"),
    "prod-historico-camara-ccjc": read_json(DATA_ROOT / "manifests" / "prod-historico-camara-ccjc.json"),
}
baseline_warnings = []
if (known_state["prod-historico-camara-plenario"] or {}).get("status") != "running":
    baseline_warnings.append("Autosave do Plenario da Camara nao esta mais em running; revisar antes de gravar.")
if (known_state["prod-historico-senado-ccj"] or {}).get("errors") != 2:
    baseline_warnings.append("A CCJ do Senado nao apresenta mais os 2 erros inventariados; revisar o novo estado.")
if (known_state["prod-historico-camara-ccjc"] or {}).get("errors") != 33:
    baseline_warnings.append("A CCJC da Camara nao apresenta mais os 33 erros inventariados; revisar o novo estado.")
if len(parquet_rows) != 6 or sum(parquet_rows.values()) != 407084:
    baseline_warnings.append("A fotografia anterior diverge de seis Parquets/407.084 textos; registrar a mudanca.")
if dataset_cutoffs.get("plenario_apartes") not in {None, "2026-05-18"}:
    baseline_warnings.append(f"O corte de apartes diverge de 2026-05-18: {dataset_cutoffs.get('plenario_apartes')}")
print("AVISOS DO BASELINE:", baseline_warnings or "nenhum")

for run in COLLECTION_RUNS:
    final_path = DATA_ROOT / "manifests" / f"{run['run_id']}.json"
    autosave_path = DATA_ROOT / "manifests" / f"{run['run_id']}.autosave.json"
    unresolved = {}
    for source in run.get("checkpoint_sources") or [run["source"]]:
        checkpoint = read_json(DATA_ROOT / "checkpoints" / source / f"{run['dataset']}.json") or {}
        current = (checkpoint.get("runs") or {}).get(run["run_id"], {}) or {}
        failed = set((current.get("failed_partitions") or {}).keys())
        completed = set((current.get("completed_partitions") or {}).keys())
        unresolved[source] = sorted(failed - completed)
    print(run["key"], "final=", compact_manifest(final_path) if final_path.exists() else None,
          "autosave=", compact_manifest(autosave_path) if autosave_path.exists() else None,
          "falhas_nao_resolvidas=", unresolved)

INVENTORY_REPORT = {
    "cycle_id": EXPECTED_CYCLE_ID,
    "data_root": str(DATA_ROOT),
    "manifest_count": len(manifest_paths),
    "dataset_cutoffs": dataset_cutoffs,
    "previous_text_parquets": parquet_rows,
    "previous_text_rows": sum(parquet_rows.values()),
    "known_state": known_state,
    "warnings": baseline_warnings,
}

## Gravar controle

Revise o inventario. A celula recusa sobrescrever outro ciclo e exige que o `cycle_id` seja digitado.

In [ ]:
GRAVAR_CONFIGURACAO = False
CONFIRMAR_CICLO = ""  # Digite 20260713.
SOBRESCREVER_MESMO_CICLO = False

if GRAVAR_CONFIGURACAO:
    assert CONFIRMAR_CICLO == EXPECTED_CYCLE_ID
    existing = read_json(ACTIVE_CONFIG_PATH)
    if existing:
        assert existing.get("cycle_id") == EXPECTED_CYCLE_ID, f"Outro ciclo esta ativo: {existing.get('cycle_id')}"
        assert SOBRESCREVER_MESMO_CICLO, "Ative SOBRESCREVER_MESMO_CICLO apos revisar o active.json existente."
    CYCLE_DIR.mkdir(parents=True, exist_ok=True)
    serialized = json.dumps(CONFIG_CANDIDATE, ensure_ascii=False, indent=2, sort_keys=True) + "\n"
    ACTIVE_CONFIG_PATH.write_text(serialized, encoding="utf-8")
    (CYCLE_DIR / "config.json").write_text(serialized, encoding="utf-8")
    audit_dir = CYCLE_DIR / "audits"
    audit_dir.mkdir(parents=True, exist_ok=True)
    (audit_dir / "initial_inventory.json").write_text(
        json.dumps(INVENTORY_REPORT, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print("Controle gravado:", ACTIVE_CONFIG_PATH)
    print("Copia do ciclo:", CYCLE_DIR / "config.json")
else:
    print("Somente auditoria: GRAVAR_CONFIGURACAO=False")